# PySpark — Window Functions

Window functions compute a value for each row **based on a group of related rows** — without collapsing the group into a single row like `groupBy` does.

Think of it as: "For each row, look at the rows *around it* (the window) and compute something."

```python
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, lag, lead, row_number, avg

window = Window.partitionBy("dept").orderBy(col("salary").desc())
df.withColumn("rank", rank().over(window))
```

| Function | Purpose |
|----------|---------|
| `row_number()` | Sequential 1,2,3… within partition — no ties |
| `rank()` | Rank with gaps on ties: 1,1,3 |
| `dense_rank()` | Rank without gaps: 1,1,2 |
| `lag(col, n)` | Value N rows **before** current row |
| `lead(col, n)` | Value N rows **after** current row |
| `avg() / sum() over window` | Running or partition-scoped aggregation |

## Setup

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col, rank, dense_rank, row_number,
    lag, lead, avg, sum as spark_sum, max as spark_max,
    round as spark_round, ntile
)

active = SparkSession.getActiveSession()
if active:
    active.stop()

spark = SparkSession.builder \
    .appName("WindowFunctions") \
    .config("spark.pyspark.python", sys.executable) \
    .getOrCreate()

data = [
    ("Alice",   "Engineering", 2023, 1, 95000),
    ("Bob",     "Marketing",   2023, 1, 72000),
    ("Charlie", "Engineering", 2023, 1, 110000),
    ("Diana",   "HR",          2023, 1, 65000),
    ("Eve",     "Marketing",   2023, 1, 85000),
    ("Frank",   "Engineering", 2023, 1, 95000),
]
df = spark.createDataFrame(data, ["name", "dept", "year", "month", "salary"])
df.show()

## 1. Defining a Window Spec

A window specification has two key parts:
- `partitionBy()` — divides data into groups (like GROUP BY)
- `orderBy()` — orders rows within each partition

In [ ]:
# Window: group by department, order by salary descending
dept_window = Window.partitionBy("dept").orderBy(col("salary").desc())

# Global window (no partition) — order across all rows
global_window = Window.orderBy(col("salary").desc())

print("Window specs defined.")

## 2. row_number(), rank(), dense_rank()

Three ways to rank rows — differ only in how they handle **ties**.

In [ ]:
dept_window = Window.partitionBy("dept").orderBy(col("salary").desc())

df_ranked = df.withColumn("row_num",    row_number().over(dept_window)) \
              .withColumn("rank",        rank().over(dept_window)) \
              .withColumn("dense_rank",  dense_rank().over(dept_window))

df_ranked.select("name", "dept", "salary", "row_num", "rank", "dense_rank") \
         .orderBy("dept", "rank") \
         .show()

print("Alice and Frank both earn 95000 in Engineering — same salary, so tied.")
print("row_num : 1,2,3  (arbitrary tiebreak, no gaps)")
print("rank    : 1,1,3  (both get 1, next is 3 — gap)")
print("dense_rank: 1,1,2 (both get 1, next is 2 — no gap)")

## 3. lag() and lead() — Access Adjacent Row Values

`lag(col, n)` — look N rows **backward**  
`lead(col, n)` — look N rows **forward**  
Default value can be specified for rows with no previous/next row.

In [ ]:
# Time-series sales data — track month-over-month changes
sales_data = [
    ("Alice", 2024, 1,  45000),
    ("Alice", 2024, 2,  52000),
    ("Alice", 2024, 3,  48000),
    ("Alice", 2024, 4,  61000),
    ("Bob",   2024, 1,  30000),
    ("Bob",   2024, 2,  28000),
    ("Bob",   2024, 3,  35000),
    ("Bob",   2024, 4,  42000),
]
df_sales = spark.createDataFrame(sales_data, ["name", "year", "month", "sales"])

# Window: per person, ordered by month
person_time_window = Window.partitionBy("name").orderBy("year", "month")

df_mom = df_sales \
    .withColumn("prev_month_sales", lag("sales", 1, 0).over(person_time_window)) \
    .withColumn("next_month_sales", lead("sales", 1).over(person_time_window)) \
    .withColumn("mom_change",       col("sales") - col("prev_month_sales")) \
    .withColumn("mom_pct",          spark_round((col("sales") - col("prev_month_sales")) / col("prev_month_sales") * 100, 1))

df_mom.show()

## 4. Aggregations Over a Window

`avg()`, `sum()`, `max()` etc. can be applied over a window — they compute per-group without collapsing rows.

In [ ]:
dept_window_ord = Window.partitionBy("dept").orderBy(col("salary").desc())
dept_window_all = Window.partitionBy("dept")   # unordered — whole partition

df_agg = df \
    .withColumn("dept_avg_salary",  spark_round(avg("salary").over(dept_window_all), 0)) \
    .withColumn("dept_total_salary", spark_sum("salary").over(dept_window_all)) \
    .withColumn("dept_max_salary",   spark_max("salary").over(dept_window_all)) \
    .withColumn("pct_of_dept_total", spark_round(col("salary") / spark_sum("salary").over(dept_window_all) * 100, 1))

df_agg.select("name", "dept", "salary", "dept_avg_salary", "pct_of_dept_total") \
      .orderBy("dept", col("salary").desc()) \
      .show()

## 5. Running Totals — Frame Specification

By default an ordered window aggregation accumulates up to the current row.  
Use `rowsBetween()` to control the frame explicitly.

In [ ]:
# Running total (cumulative sum) of sales per person
person_ord_window = Window.partitionBy("name").orderBy("year", "month")

# Default for ordered window: unbounded preceding to current row
running_total_window = Window.partitionBy("name") \
    .orderBy("year", "month") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_sales.withColumn(
    "cumulative_sales",
    spark_sum("sales").over(running_total_window)
).show()

# 3-month moving average
moving_avg_window = Window.partitionBy("name") \
    .orderBy("year", "month") \
    .rowsBetween(-2, Window.currentRow)   # current row + 2 rows back

df_sales.withColumn(
    "3m_avg",
    spark_round(avg("sales").over(moving_avg_window), 0)
).show()

## 6. ntile() — Split into N Equal Buckets

In [ ]:
# Split employees into salary quartiles (4 groups)
all_window = Window.orderBy(col("salary").desc())

df.withColumn("quartile", ntile(4).over(all_window)) \
  .select("name", "dept", "salary", "quartile") \
  .orderBy("quartile", col("salary").desc()) \
  .show()

print("ntile(4) → salary quartiles: 1=top 25%, 2=next 25%, etc.")

## 7. Real-World: Top Earner per Department

In [ ]:
# Classic interview question: get the top N earners per department
dept_salary_window = Window.partitionBy("dept").orderBy(col("salary").desc())

top1_per_dept = df \
    .withColumn("rank", rank().over(dept_salary_window)) \
    .filter(col("rank") == 1) \
    .select("dept", "name", "salary")

print("Top earner per department:")
top1_per_dept.show()

# Top 2 per department (handles ties correctly with dense_rank)
top2_per_dept = df \
    .withColumn("dense_rank", dense_rank().over(dept_salary_window)) \
    .filter(col("dense_rank") <= 2) \
    .select("dept", "name", "salary", "dense_rank") \
    .orderBy("dept", "dense_rank")

print("Top 2 earners per department:")
top2_per_dept.show()

## Quick Reference

```python
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, dense_rank, row_number, lag, lead, avg, sum

# Define window spec
w = Window.partitionBy("dept").orderBy(col("salary").desc())
w_all = Window.partitionBy("dept")          # unordered — whole partition
w_global = Window.orderBy(col("salary").desc())  # no partition — all rows

# Ranking
df.withColumn("rn",   row_number().over(w))    # 1,2,3 no ties
df.withColumn("rnk",  rank().over(w))          # 1,1,3 with gaps
df.withColumn("drnk", dense_rank().over(w))    # 1,1,2 no gaps

# Offset
df.withColumn("prev", lag("col", 1, 0).over(w))   # prior row value
df.withColumn("next", lead("col", 1).over(w))      # next row value

# Aggregation (no collapse)
df.withColumn("dept_avg", avg("salary").over(w_all))
df.withColumn("running",  sum("sales").over(w))     # cumulative (default ordered frame)

# Quartiles
from pyspark.sql.functions import ntile
df.withColumn("quartile", ntile(4).over(w))

# Frame specification
w_3row = w.rowsBetween(-2, Window.currentRow)      # current + 2 rows back
w_all_rows = w.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
```

## Interview Questions

1. What is the difference between `groupBy().agg()` and a window function?
2. What is the difference between `rank()`, `dense_rank()`, and `row_number()`? How do they handle ties?
3. What is `partitionBy()` in a window spec? How is it different from `groupBy()`?
4. What does `lag()` return for the first row in a partition?
5. How would you find the top-N earner per department?
6. What is a running total and how do you compute one with window functions?
7. What is `rowsBetween()` used for?

## Practice Exercises

**Easy:**
1. Rank employees by salary within each department (highest = rank 1) using `rank()`.
2. Add a column `row_num` that numbers employees 1,2,3... within each department ordered by name.

**Medium:**
3. For each employee, add a column `prev_salary` showing the salary of the previous employee (by id) in the same department.
4. Compute the running total of salaries ordered by salary descending.
5. Add a column `pct_of_dept` showing each employee's salary as a percentage of their department total.

**Advanced:**
6. Using the sales time-series data, compute a 3-month moving average per person.
7. Find all employees who are in the top 25% of earners within their department (use `ntile(4)`).
8. For each employee, show their salary vs the department average and flag if they earn more than 1.5x the average.

In [ ]:
spark.stop()